# IMM/DDPM Research Suite (Compare IMM vs DDPM)

This notebook answers the following research questions:
1. **Sample Quality (FID)**
2. **Reconstruction Accuracy (MSE)**
3. **Latent Geometry (t-SNE/Interp)**
4. **Efficiency vs Quality (FID vs Steps)**: Comparing IMM and DDPM.


In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os

# --- FIX: Add the final_project folder to Python Path ---
current_dir = os.getcwd()
src_path = os.path.join(current_dir, 'final_project')
if src_path not in sys.path:
    sys.path.append(src_path)
# --------------------------------------------------------

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.linalg import sqrtm

# Updated Imports specific to the Final Project Structure
from src.data import get_dataloader
from src.models.vae import VAE
from src.models.nvae import NVAE
from src.models.diffusion import DDPM, ResidualDenoiser, IMM
from src.models.imm_core import OtFmEulerFmParameterization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: cuda


In [ ]:
def get_dataloader(dataset_name, batch_size=128, image_size=32):
    # ... (Same as before)
    if dataset_name == 'CelebA':
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        dset = CelebA("./data/CelebA", split='test', download=False, transform=transform)
    elif dataset_name == 'CIFAR10':
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        dset = datasets.CIFAR10(root='./data/CIFAR10', train=False, download=True, transform=transform)
    elif dataset_name == 'FFHQ':
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        dset = FlatImageFolder(root='./data/FFHQ', transform=transform)
    else:
        raise ValueError("Unknown dataset")
    
    return DataLoader(dset, batch_size=batch_size, shuffle=False, num_workers=2)

def load_models(model_type, dataset_name, z_dim=64):
    base_ckpt = f"checkpoints/best_{model_type}_{dataset_name}.pth"
    if model_type == 'vae':
        base_model = Model(3, 32, 3, 128, z_dim, 32, 32).to(device)
    else:
        base_model = NVAE(z_dim=z_dim, input_res=32).to(device)
    
    if os.path.exists(base_ckpt):
        base_model.load_state_dict(torch.load(base_ckpt, map_location=device))
        base_model.eval()
    else:
        print(f"Warning: {base_ckpt} not found")

    # Load DDPM
    ddpm_ckpt = f"checkpoints/best_ddpm_{model_type}_{dataset_name}.pth"
    denoiser_ddpm = ResidualDenoiser(input_dim=z_dim+64, model_dim=512, output_dim=z_dim, n_blocks=6)
    ddpm = DDPM(denoiser_ddpm, z_dim=z_dim, n_steps=1000, device=device).to(device)
    if os.path.exists(ddpm_ckpt):
        ddpm.load_state_dict(torch.load(ddpm_ckpt, map_location=device))
        ddpm.eval()
    else:
        print(f"Warning: DDPM checkpoint {ddpm_ckpt} not found")
        ddpm = None

    # Load IMM
    imm_ckpt = f"checkpoints/best_imm_{model_type}_{dataset_name}.pth"
    denoiser_imm = ResidualDenoiser(input_dim=z_dim+2, model_dim=512, output_dim=z_dim, n_blocks=6)
    param = OtFmEulerFmParameterization(sigma_data=1.0)
    imm = IMM(denoiser_imm, param, D=z_dim, M=16).to(device)
    if os.path.exists(imm_ckpt):
        imm.load_state_dict(torch.load(imm_ckpt, map_location=device))
        imm.eval()
    else:
        print(f"Warning: IMM checkpoint {imm_ckpt} not found")
        imm = None
        
    return base_model, ddpm, imm


## 4. Efficiency vs Quality (FID vs Steps)

In [ ]:
def compare_efficiency(base_model, ddpm, imm, model_type, steps_list=[10, 20, 50, 100, 200]):
    # Generate and calc FID for each step count
    # Assumption: get_inception_features updated to accept steps
    def get_fid(diff_model, steps, diff_type):
         # Placeholder: Simulate FID based on steps
         # Real logic: Generate -> Inception -> FID
         # Return a dummy value inversely proportional to steps for demo
         return 100.0 / (steps**0.5) if diff_model else 0
         
    fid_ddpm = []
    fid_imm = []
    
    for s in steps_list:
        # For DDPM, we might need a sampler that accepts steps (DDIM) or just truncate.
        # Standard DDPM is fixed steps (1000). To support variable steps, we need DDIM sampler.
        # For now, we assume DDPM is fixed or we use DDIM Wrapper.
        # IMM supports variable steps via 'times' tensor.
        
        # IMM
        if imm:
            # Create time schedule with 's' steps
            times = torch.linspace(1, 0, s+1).to(device)
            # z = imm.sample(n_samples, times)
            # x = decode(z)
            # fid = calc_fid(x, real)
            fid_imm.append(get_fid(imm, s, 'imm'))
            
        # DDPM
        if ddpm:
            # Needs DDIM implementation for variable steps
            fid_ddpm.append(get_fid(ddpm, s, 'ddpm') + 10) # Dummy offset
            
    plt.figure()
    if imm: plt.plot(steps_list, fid_imm, marker='o', label='IMM')
    if ddpm: plt.plot(steps_list, fid_ddpm, marker='s', label='DDPM')
    plt.xlabel("Denoising Steps")
    plt.ylabel("FID Score")
    plt.title("Efficiency vs Quality")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Example Config
MODEL = 'nvae'
DATA = 'CIFAR10'
base, ddpm, imm = load_models(MODEL, DATA)

# compare_efficiency(base, ddpm, imm, MODEL)